In [1]:
import os
import time
from datetime import datetime, timezone

from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StructType, StructField, StringType


In [2]:
# ============================================================
# 1. CONFIGURATION
# ============================================================
KAFKA_BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka1:19092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC", "retail-sales")
STARTING_OFFSETS = os.getenv("KAFKA_STARTING_OFFSETS", "latest")     # only used the very first time (no checkpoint yet)
MAX_OFFSETS = os.getenv("MAX_OFFSETS_PER_TRIGGER", "2000")
TRIGGER_INTERVAL = os.getenv("STREAM_TRIGGER", "10 seconds")
SPARK_VERSION = os.getenv("SPARK_KAFKA_VERSION", "3.1.2")            # the Kafka connector must match Spark exactly
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:" + SPARK_VERSION

LAKE_OS = os.getenv("LAKE_ROOT", "/data-lake/data-lake")             # path for python (os.*)
LAKE = "file://" + LAKE_OS.rstrip("/")                                           # path for Spark: always the local disk, never HDFS
SILVER = LAKE + "/silver"
SILVER_STREAM = SILVER + "/streaming/retail_sales"
REJECTED_STREAM = SILVER + "/_rejected/streaming_retail_sales"
GOLD = LAKE + "/gold"
GOLD_STREAM = GOLD + "/streaming"
GOLD_UNIFIED = GOLD + "/unified"
MONITORING = LAKE + "/monitoring/streaming_retail_sales"
CHECKPOINT = LAKE + "/checkpoints/retail_sales_stream_final"

spark = None            # created in main()
c = F.col

# (dimension table in Silver, key column in the dimension, key column in the event, label columns)
DIMENSIONS = [
    ("DimProduct", "ProductKey", "ProductID", ["ProductName", "BrandName", "ClassName"]),
    ("DimCustomer", "CustomerKey", "CustomerID", ["FullName", "CustomerType"]),
    ("DimStore", "StoreKey", "StoreID", ["StoreName", "StoreType"]),
]
ORPHAN_REASONS = {"ProductID": "PRODUCT_NOT_FOUND", "CustomerID": "CUSTOMER_NOT_FOUND", "StoreID": "STORE_NOT_FOUND"}

SOURCE_SCHEMA = StructType([
    StructField("EventID", StringType(), True), StructField("EventType", StringType(), True),
    StructField("EventTime", StringType(), True), StructField("CustomerID", StringType(), True),
    StructField("ProductID", StringType(), True), StructField("StoreID", StringType(), True),
    StructField("Quantity", StringType(), True), StructField("Amount", StringType(), True),
    StructField("Source", StringType(), True), StructField("_corrupt_record", StringType(), True),
])
REJECT_COLUMNS = ["EventID", "EventType", "EventTime", "CustomerID", "ProductID", "StoreID", "Quantity", "Amount",
                  "Source", "_topic", "_partition", "_offset", "_kafka_timestamp", "_raw"]
SILVER_COLUMNS = ["EventID", "EventType", "EventTimeParsed", "SaleYear", "SaleMonth", "CustomerID", "ProductID", "StoreID",
                  "Quantity", "Amount", "NetRevenue", "Source", "ProductName", "BrandName", "ClassName", "FullName",
                  "CustomerType", "StoreName", "StoreType", "_topic", "_partition", "_offset", "_kafka_timestamp"]



In [3]:
# ============================================================
# 2. HELPERS
# ============================================================
def os_path(path):
    return path[len("file://"):] if path.startswith("file://") else path


def has_parquet(path):
    for _, _, files in os.walk(os_path(path)):
        if any(name.endswith(".parquet") for name in files):
            return True
    return False


def read_if_exists(path):
    """plain read: keeps partition columns such as SaleYear (do NOT use recursiveFileLookup here, it drops them)"""
    return spark.read.parquet(path) if has_parquet(path) else None


def read_stream_silver():
    """all accepted batches: .../retail_sales/batch_*  (a glob, because Spark cannot read a folder of plain sub-folders)"""
    return spark.read.parquet(SILVER_STREAM + "/batch_*") if has_parquet(SILVER_STREAM) else None


def write_one(df, path):
    """small output: one file, replaced when the batch is replayed"""
    df.coalesce(1).write.mode("overwrite").parquet(path)


def load_dimension(table, dim_key, event_key, labels):
    """The dimension MUST be readable and its key unique; otherwise stop with an error instead of labelling everything 'Unknown'."""
    path = SILVER + "/" + table
    if not has_parquet(path):
        raise RuntimeError("Dimension not found: %s  (run 02_silver.py first)" % path)
    df = spark.read.parquet(path)
    missing = [x for x in [dim_key] + labels if x not in df.columns]
    if missing:
        raise RuntimeError("%s is missing columns %s" % (table, missing))
    selected = (df.select(c(dim_key).cast("int").alias("_dim_" + event_key), *[c(x) for x in labels])
                  .filter(c("_dim_" + event_key).isNotNull()))
    if selected.groupBy("_dim_" + event_key).count().filter(c("count") > 1).limit(1).count():
        raise RuntimeError("%s has duplicate %s values: fix the dimension grain / SCD2 key mapping before streaming." % (table, dim_key))
    return selected.withColumn("_found_" + event_key, F.lit(True))


def enrich_events(events):
    """left-join the 3 dimensions and add _orphan_reason for events whose key does not exist in its dimension"""
    for table, dim_key, event_key, labels in DIMENSIONS:
        dim = F.broadcast(load_dimension(table, dim_key, event_key, labels))
        events = events.join(dim, events[event_key] == dim["_dim_" + event_key], "left").drop("_dim_" + event_key)
    reason = None
    for table, dim_key, event_key, labels in DIMENSIONS:
        cond = c("_found_" + event_key).isNull()
        reason = F.when(cond, ORPHAN_REASONS[event_key]) if reason is None else reason.when(cond, ORPHAN_REASONS[event_key])
    events = events.withColumn("_orphan_reason", reason)
    return events.drop(*["_found_" + d[2] for d in DIMENSIONS])


In [4]:
# ============================================================
# 3. GOLD
# ============================================================
def refresh_gold(batch_id):
    events = read_stream_silver()
    if events is None:
        print("[GOLD] no accepted events yet")
        return

    top_products = (events.groupBy("ProductID", "ProductName", "BrandName", "ClassName")
                    .agg(F.sum("Quantity").alias("TotalQuantity"), F.sum("NetRevenue").alias("TotalRevenue"),
                         F.countDistinct("EventID").alias("TotalEvents"))
                    .withColumn("RevenueRank", F.dense_rank().over(Window.orderBy(F.desc("TotalRevenue")))))

    customer_summary = (events.groupBy("CustomerID", "FullName", "CustomerType")
                        .agg(F.sum("NetRevenue").alias("TotalSpent"), F.sum("Quantity").alias("TotalUnits"),
                             F.countDistinct("EventID").alias("TotalEvents"),
                             F.min("EventTimeParsed").alias("FirstEvent"), F.max("EventTimeParsed").alias("LastEvent")))

    monthly_brand = (events.groupBy("SaleYear", "SaleMonth", "BrandName")
                     .agg(F.sum("NetRevenue").alias("Revenue"), F.sum("Quantity").alias("Units"),
                          F.countDistinct("EventID").alias("Events")))

    store_monthly = (events.groupBy("SaleYear", "SaleMonth", "StoreID", "StoreName", "StoreType")
                     .agg(F.sum("NetRevenue").alias("Revenue"), F.sum("Quantity").alias("Units"),
                          F.countDistinct("EventID").alias("Events")))

    top_customers = (customer_summary.withColumn("CustomerRank", F.row_number().over(Window.orderBy(F.desc("TotalSpent"))))
                     .filter(c("CustomerRank") <= 10))

    brand_year = events.groupBy("SaleYear", "BrandName").agg(F.sum("NetRevenue").alias("BrandRevenue"))
    brand_year = (brand_year.withColumn("YearRevenue", F.sum("BrandRevenue").over(Window.partitionBy("SaleYear")))
                  .withColumn("RevenueShare", F.when(c("YearRevenue") != 0, c("BrandRevenue") / c("YearRevenue")).otherwise(F.lit(0))))

    sales_per_minute = (events.groupBy(F.window("EventTimeParsed", "1 minute").alias("w"))
                        .agg(F.sum("NetRevenue").alias("Revenue"), F.sum("Quantity").alias("Units"),
                             F.countDistinct("EventID").alias("Events"))
                        .select(c("w.start").alias("WindowStart"), c("w.end").alias("WindowEnd"), "Revenue", "Units", "Events"))

    for name, df in [("streaming_top_products", top_products), ("streaming_customer_purchase_frequency", customer_summary),
                     ("streaming_monthly_sales_by_brand", monthly_brand), ("streaming_store_monthly_kpi", store_monthly),
                     ("streaming_top_customers", top_customers), ("streaming_brand_share_by_year", brand_year),
                     ("streaming_sales_per_minute", sales_per_minute)]:
        write_one(df, GOLD_STREAM + "/" + name)
    print("[GOLD] streaming datasets refreshed after batch %d" % batch_id)

    refresh_unified(events)


def batch_gold(name):
    return read_if_exists(GOLD + "/" + name)


def require(df, name, columns):
    """stop with a clear message instead of silently dropping the batch history"""
    missing = [x for x in columns if x not in df.columns]
    if missing:
        raise RuntimeError("batch Gold '%s' is missing %s (it has %s)" % (name, missing, df.columns))


def refresh_unified(events):
    """Batch Gold (history) + streaming events (live) in one place. The batch Gold folders are only read."""
    dec = "decimal(29,4)"
    if batch_gold("top_products") is None:
        print("[UNIFIED] batch Gold not found -> unified datasets contain streaming data only")

    # --- 1) brand x month : batch rows + streaming rows (column data_source tells them apart)
    live = (events.groupBy("SaleYear", "SaleMonth", "BrandName")
            .agg(F.sum("Quantity").cast("long").alias("units"), F.sum("NetRevenue").cast(dec).alias("revenue"),
                 F.countDistinct("EventID").cast("long").alias("orders"))
            .withColumn("data_source", F.lit("stream")))
    hist = batch_gold("monthly_sales_by_brand")
    if hist is not None:
        require(hist, "monthly_sales_by_brand", ["SaleYear", "SaleMonth", "BrandName", "units_sold", "revenue", "orders"])
        hist = hist.select(c("SaleYear").cast("int"), c("SaleMonth").cast("int"), "BrandName",
                           c("units_sold").cast("long").alias("units"), c("revenue").cast(dec),
                           c("orders").cast("long"), F.lit("batch").alias("data_source"))
        live = hist.unionByName(live)
    write_one(live, GOLD_UNIFIED + "/sales_by_brand_month")

    # --- 2) store x month : same idea
    live = (events.groupBy("SaleYear", "SaleMonth", c("StoreID").alias("StoreKey"), "StoreName")
            .agg(F.sum("NetRevenue").cast(dec).alias("revenue"), F.countDistinct("EventID").cast("long").alias("orders"))
            .withColumn("data_source", F.lit("stream")))
    hist = batch_gold("store_monthly_kpi")
    if hist is not None:
        require(hist, "store_monthly_kpi", ["SaleYear", "SaleMonth", "StoreKey", "StoreName", "revenue", "orders"])
        hist = hist.select(c("SaleYear").cast("int"), c("SaleMonth").cast("int"), c("StoreKey").cast("int"), "StoreName",
                           c("revenue").cast(dec), c("orders").cast("long"), F.lit("batch").alias("data_source"))
        live = hist.unionByName(live)
    write_one(live, GOLD_UNIFIED + "/store_month_overall")

    # --- 3) products : one row per product with batch, stream and total
    live = (events.groupBy(c("ProductID").alias("ProductKey"))
            .agg(F.max("ProductName").alias("ProductName"), F.max("BrandName").alias("BrandName"),
                 F.max("ClassName").alias("ClassName"), F.sum("Quantity").cast("long").alias("units"),
                 F.sum("NetRevenue").cast(dec).alias("revenue"))
            .withColumn("data_source", F.lit("stream")))
    hist = batch_gold("top_products")
    if hist is not None:
        require(hist, "top_products", ["ProductKey", "ProductName", "BrandName", "ClassName", "units_sold", "revenue"])
        hist = hist.select(c("ProductKey").cast("int"), "ProductName", "BrandName", "ClassName",
                           c("units_sold").cast("long").alias("units"), c("revenue").cast(dec),
                           F.lit("batch").alias("data_source"))
        live = hist.unionByName(live)
    zero = F.lit(0).cast(dec)
    products = (live.groupBy("ProductKey")
                .agg(F.max("ProductName").alias("ProductName"), F.max("BrandName").alias("BrandName"),
                     F.max("ClassName").alias("ClassName"),
                     F.sum(F.when(c("data_source") == "batch", c("units")).otherwise(0)).alias("batch_units"),
                     F.sum(F.when(c("data_source") == "stream", c("units")).otherwise(0)).alias("stream_units"),
                     F.sum(F.when(c("data_source") == "batch", c("revenue")).otherwise(zero)).alias("batch_revenue"),
                     F.sum(F.when(c("data_source") == "stream", c("revenue")).otherwise(zero)).alias("stream_revenue"))
                .withColumn("total_units", c("batch_units") + c("stream_units"))
                .withColumn("total_revenue", c("batch_revenue") + c("stream_revenue"))
                .withColumn("revenue_rank", F.dense_rank().over(Window.orderBy(F.desc("total_revenue")))))
    write_one(products, GOLD_UNIFIED + "/top_products_overall")

    # --- 4) customers
    live = (events.groupBy(c("CustomerID").alias("CustomerKey"))
            .agg(F.max("FullName").alias("FullName"), F.max("CustomerType").alias("CustomerType"),
                 F.countDistinct("EventID").cast("long").alias("orders"), F.sum("NetRevenue").cast(dec).alias("spent"))
            .withColumn("data_source", F.lit("stream")))
    hist = batch_gold("customer_purchase_frequency")
    if hist is not None:
        require(hist, "customer_purchase_frequency", ["CustomerKey", "FullName", "CustomerType", "orders", "total_spent"])
        hist = hist.select(c("CustomerKey").cast("int"), "FullName", "CustomerType", c("orders").cast("long"),
                           c("total_spent").cast(dec).alias("spent"), F.lit("batch").alias("data_source"))
        live = hist.unionByName(live)
    customers = (live.groupBy("CustomerKey")
                 .agg(F.max("FullName").alias("FullName"), F.max("CustomerType").alias("CustomerType"),
                      F.sum(F.when(c("data_source") == "batch", c("orders")).otherwise(0)).alias("batch_orders"),
                      F.sum(F.when(c("data_source") == "stream", c("orders")).otherwise(0)).alias("stream_orders"),
                      F.sum(F.when(c("data_source") == "batch", c("spent")).otherwise(zero)).alias("batch_spent"),
                      F.sum(F.when(c("data_source") == "stream", c("spent")).otherwise(zero)).alias("stream_spent"))
                 .withColumn("total_orders", c("batch_orders") + c("stream_orders"))
                 .withColumn("total_spent", c("batch_spent") + c("stream_spent"))
                 .withColumn("spend_rank", F.dense_rank().over(Window.orderBy(F.desc("total_spent")))))
    write_one(customers, GOLD_UNIFIED + "/customer_spend_overall")

    # --- 5) one-row overview: this is the "KPI" that moves when a new sale arrives
    totals = events.agg(F.sum("NetRevenue"), F.countDistinct("EventID"), F.max("EventTimeParsed")).first()
    stream_revenue = float(totals[0] or 0)
    batch_revenue = float(hist_total("top_products", "revenue"))
    overview = spark.createDataFrame(
        [(batch_revenue, stream_revenue, batch_revenue + stream_revenue, int(totals[1] or 0),
          str(totals[2]), datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"))],
        "batch_revenue double, stream_revenue double, total_revenue double, stream_events long, "
        "last_event_time string, refreshed_at_utc string")
    write_one(overview, GOLD_UNIFIED + "/kpi_overview")
    print("[UNIFIED] unified datasets refreshed (batch revenue %.2f + stream revenue %.2f)" % (batch_revenue, stream_revenue))


def hist_total(name, column):
    df = batch_gold(name)
    return 0 if df is None else (df.agg(F.sum(column)).first()[0] or 0)


In [5]:
# ============================================================
# 4. MICRO-BATCH
# ============================================================
def write_metrics(batch_id, input_count, invalid, duplicates, orphans, accepted):
    row = [(int(batch_id), int(input_count), int(invalid), int(duplicates), int(orphans), int(accepted),
            datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"))]
    write_one(spark.createDataFrame(
        row, "batch_id long, input_events long, invalid_events long, duplicate_events long, orphan_events long, "
             "accepted_events long, processed_at_utc string"), "%s/batch_%08d" % (MONITORING, batch_id))


def process_microbatch(kafka_batch, batch_id):
    started = time.time()
    batch_id = int(batch_id)
    silver_target = "%s/batch_%08d" % (SILVER_STREAM, batch_id)

    if has_parquet(silver_target):
        # the batch was fully written before a restart: do not recompute, just make sure Gold is up to date
        print("[RECOVERY] batch %d is already in Silver" % batch_id)
        refresh_gold(batch_id)
        return

    kafka_batch = kafka_batch.persist()
    try:
        input_count = kafka_batch.count()
        if input_count == 0:
            print("[BATCH %d] empty" % batch_id)
            return

        # ---- parse + normalize
        parsed = (kafka_batch.select(c("value").cast("string").alias("_raw"), c("topic").alias("_topic"),
                                     c("partition").alias("_partition"), c("offset").alias("_offset"),
                                     c("timestamp").alias("_kafka_timestamp"))
                  .withColumn("_event", F.from_json(c("_raw"), SOURCE_SCHEMA,
                                                    {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"}))
                  .select("_raw", "_topic", "_partition", "_offset", "_kafka_timestamp", c("_event").alias("_event_struct"), "_event.*"))
        df = (parsed
              .withColumn("EventID", F.trim("EventID"))
              .withColumn("EventType", F.upper(F.trim("EventType")))
              .withColumn("EventTimeParsed", F.to_timestamp("EventTime"))
              .withColumn("CustomerID", c("CustomerID").cast("int"))
              .withColumn("ProductID", c("ProductID").cast("int"))
              .withColumn("StoreID", c("StoreID").cast("int"))
              .withColumn("Quantity", c("Quantity").cast("int"))
              .withColumn("Amount", c("Amount").cast("decimal(19,4)"))
              .withColumn("Source", F.trim("Source")))

        # ---- validation: the first failing rule gives the reason
        rules = [
            (c("_event_struct").isNull() | c("_corrupt_record").isNotNull(), "MALFORMED_JSON"),
            (c("EventID").isNull() | (c("EventID") == ""), "MISSING_EVENT_ID"),
            (c("EventType").isNull() | (~c("EventType").isin("SALE", "SALECOMPLETED")), "INVALID_EVENT_TYPE"),
            (c("EventTimeParsed").isNull(), "INVALID_EVENT_TIME"),
            (c("CustomerID").isNull() | c("ProductID").isNull() | c("StoreID").isNull(), "INVALID_DIMENSION_ID"),
            ((c("CustomerID") <= 0) | (c("ProductID") <= 0) | (c("StoreID") <= 0), "NON_POSITIVE_DIMENSION_ID"),
            (c("Quantity").isNull() | (c("Quantity") <= 0), "INVALID_QUANTITY"),
            (c("Amount").isNull() | (c("Amount") < 0), "INVALID_AMOUNT"),
        ]
        reason = None
        for cond, text in rules:
            reason = F.when(cond, text) if reason is None else reason.when(cond, text)
        flagged = df.withColumn("RejectionReason", reason)

        invalid = flagged.filter(c("RejectionReason").isNotNull())
        valid = flagged.filter(c("RejectionReason").isNull()).drop("RejectionReason")
        invalid_count = invalid.count()
        valid_count = input_count - invalid_count

        # ---- duplicates: inside the batch (keep the first one) and against everything accepted before
        first_one = Window.partitionBy("EventID").orderBy(c("_partition").asc(), c("_offset").asc())
        candidates = (valid.withColumn("_rn", F.row_number().over(first_one)).filter(c("_rn") == 1).drop("_rn"))
        before = read_stream_silver()
        if before is not None:
            candidates = candidates.join(before.select("EventID").distinct(), "EventID", "left_anti")

        # ---- referential check + labels
        enriched = enrich_events(candidates).persist()
        orphan_count = enriched.filter(c("_orphan_reason").isNotNull()).count()
        accepted = (enriched.filter(c("_orphan_reason").isNull())
                    .withColumn("NetRevenue", c("Amount"))
                    .withColumn("SaleYear", F.year("EventTimeParsed"))
                    .withColumn("SaleMonth", F.month("EventTimeParsed"))
                    .select(*SILVER_COLUMNS)
                    .withColumn("batch_id", F.lit(batch_id).cast("long"))
                    .withColumn("processed_at_utc", F.current_timestamp()))
        accepted_count = accepted.count()
        candidate_count = candidates.count()
        duplicate_count = max(0, valid_count - candidate_count)      # orphans are NOT duplicates

        # ---- rejected = invalid events + events with unknown keys
        rejected = (invalid.select(*REJECT_COLUMNS, c("RejectionReason"))
                    .unionByName(enriched.filter(c("_orphan_reason").isNotNull())
                                 .select(*REJECT_COLUMNS, c("_orphan_reason").alias("RejectionReason")))
                    .withColumn("ProcessingStage", F.lit("Silver")).withColumn("batch_id", F.lit(batch_id).cast("long")))
        if invalid_count + orphan_count > 0:
            write_one(rejected, "%s/batch_%08d" % (REJECTED_STREAM, batch_id))
        if accepted_count > 0:
            write_one(accepted, silver_target)
        write_metrics(batch_id, input_count, invalid_count + orphan_count, duplicate_count, orphan_count, accepted_count)
        enriched.unpersist()

        print("[BATCH %d] input=%d accepted=%d rejected=%d (unknown keys=%d) duplicates=%d  %.1fs"
              % (batch_id, input_count, accepted_count, invalid_count + orphan_count, orphan_count,
                 duplicate_count, time.time() - started))
        if accepted_count > 0:
            refresh_gold(batch_id)
    finally:
        kafka_batch.unpersist()



In [7]:
# ============================================================
# 5. START
# ============================================================
def build_spark():
    return (SparkSession.builder.appName("Retail-Kafka-Streaming-Pipeline").master("local[2]")
            .config("spark.jars.packages", KAFKA_PACKAGE)
            .config("spark.driver.memory", os.getenv("SPARK_DRIVER_MEMORY", "2g"))
            .config("spark.hadoop.fs.defaultFS", "file:///")           # this image may default to HDFS
            .config("spark.eventLog.enabled", "false")
            .config("spark.sql.shuffle.partitions", "4")
            .config("spark.sql.session.timeZone", "UTC")
            .getOrCreate())


def main():
    global spark
    spark = build_spark()
    spark.sparkContext.setLogLevel("WARN")
    if spark.version != SPARK_VERSION:
        raise RuntimeError("Spark is %s but the Kafka connector is for %s (set SPARK_KAFKA_VERSION)" % (spark.version, SPARK_VERSION))
    for path in (SILVER_STREAM, REJECTED_STREAM, MONITORING, GOLD_STREAM, GOLD_UNIFIED, CHECKPOINT):
        os.makedirs(os_path(path), exist_ok=True)
    print("[START] Spark %s | Kafka %s | topic %s | startingOffsets=%s (first run only)"
          % (spark.version, KAFKA_BOOTSTRAP, KAFKA_TOPIC, STARTING_OFFSETS))

    stream = (spark.readStream.format("kafka")
              .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
              .option("subscribe", KAFKA_TOPIC)
              .option("startingOffsets", STARTING_OFFSETS)
              .option("maxOffsetsPerTrigger", MAX_OFFSETS)
              .option("failOnDataLoss", "true")
              .load())
    query = (stream.writeStream.queryName("retail_sales_kafka_stream").foreachBatch(process_microbatch)
             .option("checkpointLocation", CHECKPOINT).trigger(processingTime=TRIGGER_INTERVAL).start())
    print("[STREAM] running ... Ctrl+C to stop")
    query.awaitTermination()


if __name__ == "__main__":
    main()


[START] Spark 3.1.2 | Kafka kafka1:19092 | topic retail-sales | startingOffsets=latest (first run only)
[STREAM] running ... Ctrl+C to stop
[BATCH 0] empty
[BATCH 1] input=32 accepted=32 rejected=0 (unknown keys=0) duplicates=0  11.9s
[GOLD] streaming datasets refreshed after batch 1
[UNIFIED] unified datasets refreshed (batch revenue 2718202629.81 + stream revenue 42572.88)
[BATCH 2] input=296 accepted=296 rejected=0 (unknown keys=0) duplicates=0  7.4s
[GOLD] streaming datasets refreshed after batch 2
[UNIFIED] unified datasets refreshed (batch revenue 2718202629.81 + stream revenue 369241.25)


KeyboardInterrupt: 